Plantas do Brasil: 

**Fonte:** [GBIF — Global Biodiversity Information Facility](https://www.gbif.org/)  
**Escopo:** Registros de ocorrência de plantas com flor nativas do Brasil, com coordenadas geográficas validadas.

Pipeline com quatro etapas em sequência:

1. **Coleta** — requisições paginadas à API REST do GBIF  
2. **Limpeza** — seleção de colunas, remoção de nulos, validação de coordenadas, deduplicação  
3. **Armazenamento** — exportação para `.parquet` e `.csv`  
4. **Resumo** — estatísticas descritivas do dataset final

In [1]:
import requests
import geopandas as gpd
import pandas as pd
import time
import os
from shapely.geometry import Point

### 1. Configuração

Parâmetros da busca na API e quantidade de registros alvo.  
O GBIF retorna no máximo 300 itens por página e o pipeline itera com offset até atingir o total desejado.

In [2]:
BASE_URL = "https://api.gbif.org/v1/occurrence/search"

PARAMS_BASE = {
    "kingdom":            "Plantae",
    "kingdomKey":         6,
    "country":            "BR",
    "hasCoordinate":      "true",
    "hasGeospatialIssue": "false",
    "limit":              300,
}

TOTAL_DESEJADO = 9_000

### 2. Coleta via API REST

Requisições paginadas com tratamento de erros e pausa entre chamadas para respeitar
os limites de uso da API. O loop encerra quando o total desejado é atingido ou
quando não há mais resultados disponíveis.

In [3]:
def coletar_ocorrencias(total_desejado: int) -> list[dict]:
    todos_registros = []
    offset = 0

    while len(todos_registros) < total_desejado:
        params = {**PARAMS_BASE, "offset": offset}

        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Erro na requisicao (offset={offset}): {e}")
            break

        dados = response.json()
        resultados = dados.get("results", [])

        if not resultados:
            print("Todos os registros disponiveis foram coletados.")
            break

        todos_registros.extend(resultados)

        total_disponivel = dados.get("count", "?")
        print(
            f"Pagina coletada | offset={offset:>6} | "
            f"acumulado: {len(todos_registros):>5} / {total_disponivel} disponiveis"
        )

        offset += 300
        time.sleep(0.5)

    print(f"\nColeta finalizada. Total de registros brutos: {len(todos_registros)}")
    return todos_registros

In [4]:
registros_brutos = coletar_ocorrencias(TOTAL_DESEJADO)

Pagina coletada | offset=     0 | acumulado:   300 / 5378151 disponiveis
Pagina coletada | offset=   300 | acumulado:   600 / 5378151 disponiveis
Pagina coletada | offset=   600 | acumulado:   900 / 5378151 disponiveis
Pagina coletada | offset=   900 | acumulado:  1200 / 5378151 disponiveis
Pagina coletada | offset=  1200 | acumulado:  1500 / 5378151 disponiveis
Pagina coletada | offset=  1500 | acumulado:  1800 / 5378151 disponiveis
Pagina coletada | offset=  1800 | acumulado:  2100 / 5378151 disponiveis
Pagina coletada | offset=  2100 | acumulado:  2400 / 5378151 disponiveis
Pagina coletada | offset=  2400 | acumulado:  2700 / 5378151 disponiveis
Pagina coletada | offset=  2700 | acumulado:  3000 / 5378151 disponiveis
Pagina coletada | offset=  3000 | acumulado:  3300 / 5378151 disponiveis
Pagina coletada | offset=  3300 | acumulado:  3600 / 5378151 disponiveis
Pagina coletada | offset=  3600 | acumulado:  3900 / 5378151 disponiveis
Pagina coletada | offset=  3900 | acumulado:  4200 

### 3. Limpeza e Validacao

Etapas aplicadas em sequencia:

- Selecao e renomeacao das colunas relevantes
- Remocao de registros sem coordenadas ou sem especie identificada
- Conversao de tipos numericos (`latitude`, `longitude`, `ano`, `mes`)
- Filtragem pelo bounding box geografico do Brasil
- Validacao espacial das coordenadas utilizando a malha oficial dos estados brasileiros
- Correcao e preenchimento da coluna estado com base na localizacao
- Normalizacao de texto (strip + title case)
- Deduplicacao por `gbifID`

In [5]:
COLUNAS_RELEVANTES = {
    "species":          "especie",
    "family":           "familia",
    "genus":            "genero",
    "kingdom":            "reino",
    "decimalLatitude":  "latitude",
    "decimalLongitude": "longitude",
    "stateProvince":    "estado",
    "municipality":     "municipio",
    "year":             "ano",
    "month":            "mes",
    "basisOfRecord":    "tipo_registro",
    "datasetName":      "dataset",
    "gbifID":           "id_gbif",
}

In [6]:
estados = gpd.read_file("extra/BR_UF_2025.shp").to_crs("EPSG:4326")
estados.head()

,CD_UF,NM_UF,SIGLA_UF,CD_REGIAO,NM_REGIAO,SIGLA_RG,AREA_KM2,geometry
0,43,Rio Grande do Sul,RS,4,Sul,S,281707.150,"MULTIPOLYGON (((-53.52154 -33.25881, -53.51825..."
1,35,São Paulo,SP,3,Sudeste,SE,248219.485,"MULTIPOLYGON (((-48.03575 -25.35712, -48.03607..."
2,32,Espírito Santo,ES,3,Sudeste,SE,46074.440,"MULTIPOLYGON (((-40.88385 -21.16198, -40.88384..."
3,33,Rio de Janeiro,RJ,3,Sudeste,SE,43750.424,"MULTIPOLYGON (((-44.72025 -23.35934, -44.72029..."
4,41,Paraná,PR,4,Sul,S,199293.571,"MULTIPOLYGON (((-48.40723 -25.84254, -48.40732..."


In [7]:
def limpar_dados(registros_brutos: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(registros_brutos)
    print(f"Shape bruto: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

    # Selecao de colunas
    colunas_existentes = {k: v for k, v in COLUNAS_RELEVANTES.items() if k in df.columns}
    df = df[list(colunas_existentes.keys())].copy()
    df = df.rename(columns=colunas_existentes)

    # Remocao de nulos essenciais
    antes = len(df)
    df = df.dropna(subset=["latitude", "longitude", "especie"])
    print(f"Removidas {antes - len(df):,} linhas sem lat/long/especie")

    # Conversao de tipos
    df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["ano"]       = pd.to_numeric(df["ano"],        errors="coerce").astype("Int64")
    df["mes"]       = pd.to_numeric(df["mes"],        errors="coerce").astype("Int64")

    # Bounding box do Brasil
    antes = len(df)
    df = df[
        (df["latitude"].between(-34.0, 5.5)) &
        (df["longitude"].between(-74.0, -34.0))
    ]
    print(f"Removidas {antes - len(df):,} coordenadas fora do Brasil")

    # Segunda validacao espacial
    antes = len(df)

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["longitude"],
            df["latitude"]
        ),
        crs="EPSG:4326"
    )

    estados_geo = estados.copy()

    coluna_estado = "NM_UF"

    gdf = gpd.sjoin(
        gdf,
        estados_geo[[coluna_estado, "geometry"]],
        how="inner",
        predicate="within"
    )

    removidos = antes - len(gdf)
    print(f"Removidos {removidos:,} registros fora da malha geografica")

    gdf["estado"] = gdf[coluna_estado].str.title()

    df = gdf.drop(
        columns=["geometry", "index_right", coluna_estado],
        errors="ignore"
    )

    # Normalizacao de texto
    cols_texto = ["especie", "familia", "genero", "estado", "municipio"]
    for col in cols_texto:
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()

    # Deduplicacao
    if "id_gbif" in df.columns:
        antes = len(df)
        df = df.drop_duplicates(subset=["id_gbif"])
        print(f"Removidas {antes - len(df):,} duplicatas por id_gbif")

    df = df.reset_index(drop=True)

    print(f"Shape final: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
    print(f"Especies unicas: {df['especie'].nunique():,}")
    if "estado" in df.columns:
        print(f"Estados presentes: {df['estado'].nunique()}")

    return df

In [8]:
df = limpar_dados(registros_brutos)
df.head()

Shape bruto: 9,000 linhas x 129 colunas
Removidas 564 linhas sem lat/long/especie
Removidas 9 coordenadas fora do Brasil
Removidos 79 registros fora da malha geografica
Removidas 0 duplicatas por id_gbif
Shape final: 8,348 linhas x 13 colunas
Especies unicas: 2,835
Estados presentes: 27


,especie,familia,genero,reino,latitude,longitude,estado,municipio,ano,mes,tipo_registro,dataset,id_gbif
0,Zeyheria Tuberculosa,Bignoniaceae,Zeyheria,Plantae,-22.887250,-48.490417,São Paulo,Botucatu,2026,1,PRESERVED_SPECIMEN,NaN,3414152432
1,Mauritia Flexuosa,Arecaceae,Mauritia,Plantae,-2.704655,-42.872883,Maranhão,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938027332
2,Microgramma Squamulosa,Polypodiaceae,Microgramma,Plantae,-28.046272,-49.519303,Santa Catarina,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938029900
3,Passiflora Capsularis,Passifloraceae,Passiflora,Plantae,-23.208287,-46.966167,São Paulo,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938032403
4,Passiflora Miersii,Passifloraceae,Passiflora,Plantae,-21.769328,-46.579995,Minas Gerais,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938032781


### 4. Inspecao Rapida do Dataset

Verificacao de tipos, valores nulos e distribuicao inicial antes de salvar.

In [9]:
print(df.dtypes)
print("\n")
print(df.isnull().sum())

especie              str
familia              str
genero               str
reino                str
latitude         float64
longitude        float64
estado               str
municipio            str
ano                Int64
mes                Int64
tipo_registro        str
dataset              str
id_gbif              str
dtype: object


especie             0
familia             1
genero              0
reino               0
latitude            0
longitude           0
estado              0
municipio        7265
ano                 0
mes                 0
tipo_registro       0
dataset           943
id_gbif             0
dtype: int64


In [10]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
especie,8348,2835,Tillandsia Stricta,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN
familia,8347,219,Bromeliaceae,661,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genero,8348,1136,Vriesea,207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reino,8348,1,Plantae,8348,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,8348.0,NaN,NaN,NaN,-19.688619,7.59617,-33.513252,-24.240033,-22.43533,-15.775853,3.884186
longitude,8348.0,NaN,NaN,NaN,-45.831934,5.776532,-72.880493,-48.996177,-45.971902,-41.796731,-34.794737
estado,8348,27,São Paulo,1283,NaN,NaN,NaN,NaN,NaN,NaN,NaN
municipio,1083,115,Vila Velha,84,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ano,8348.0,<NA>,<NA>,<NA>,2026.0,0.0,2026.0,2026.0,2026.0,2026.0,2026.0
mes,8348.0,<NA>,<NA>,<NA>,1.330019,0.470248,1.0,1.0,1.0,2.0,2.0


### 5. Armazenamento

Os dados sao salvos em dois formatos:

- `.parquet` - por ser colunar e preservar a tipagem
- `.csv` - por ser legivel por qualquer ferramenta 

In [11]:
def salvar_dados(df: pd.DataFrame, pasta: str = "dados") -> None:
    os.makedirs(pasta, exist_ok=True)

    caminho_parquet = os.path.join(pasta, "plantas_brasil.parquet")
    caminho_csv     = os.path.join(pasta, "plantas_brasil.csv")

    df.to_parquet(caminho_parquet, index=False)
    df.to_csv(caminho_csv, index=False, encoding="utf-8-sig")

    kb_parquet = os.path.getsize(caminho_parquet) / 1024
    kb_csv     = os.path.getsize(caminho_csv) / 1024

In [12]:
salvar_dados(df, pasta="dados")

### 6. Resumo

Visao geral do dataset coletado

In [13]:
print(f"\nRegistros totais:  {len(df):,}")
print(f"Especies unicas:  {df['especie'].nunique():,}")

if "familia" in df.columns:
    print(f"Familias unicas:  {df['familia'].nunique():,}")

if "estado" in df.columns:
    print(f"Estados com dados:  {df['estado'].nunique()}")

if "ano" in df.columns and "mes" in df.columns:
    datas = pd.to_datetime(pd.DataFrame({
        'year': df['ano'],
        'month': df['mes'],
        'day': 1
    }))
    
    data_min = datas.min()
    data_max = datas.max()
    
    print(f"\nPeriodo dos dados:")
    print(f"  Mais antigo: {data_min.strftime('%m/%Y')}")
    print(f"  Mais recente: {data_max.strftime('%m/%Y')}")

print(f"\nTop 5 especies mais registradas:")
for especie, count in df["especie"].value_counts().head(5).items():
    print(f"  {especie:<42} {count:>5} registros")

if "estado" in df.columns:
    print(f"\nTop 5 estados com mais registros:")
    for estado, count in df["estado"].value_counts().head(5).items():
        print(f"  {estado:<30} {count:>5} registros")


Registros totais:  8,348
Especies unicas:  2,835
Familias unicas:  219
Estados com dados:  27

Periodo dos dados:
  Mais antigo: 01/2026
  Mais recente: 02/2026

Top 5 especies mais registradas:
  Tillandsia Stricta                            74 registros
  Araucaria Angustifolia                        57 registros
  Hedychium Coronarium                          57 registros
  Momordica Charantia                           54 registros
  Turnera Subulata                              53 registros

Top 5 estados com mais registros:
  São Paulo                       1283 registros
  Rio De Janeiro                  1047 registros
  Paraná                           886 registros
  Rio Grande Do Sul                801 registros
  Minas Gerais                     751 registros
